# CSE 5526 — Programming Assignment #1 (PA1)

**STUDENT INSTRUCTION NOTEBOOK**

---

⚠️ **IMPORTANT**
- You may ONLY modify code blocks marked with **`TODO`**
- Do **NOT** change function names, inputs, outputs, or print/plot formats
- Your notebook **must be run before submission**
- Follow the specifications exactly

---

### What you are building

A three-layer network (64 → 32 → 16 → 10) trained from scratch with per-sample SGD on the
8×8 handwritten digits dataset, evaluated with 5-fold stratified cross-validation.

You implement two output layers: **sigmoid with scaled sum-of-squared-error** (parts 1–3) and
**softmax with cross-entropy** (part 4). Both share the same hidden layers and the same training
loop — only the output activation and the output-layer error term differ.

### Map of this notebook

| Section | Assignment part | Contents |
|---|---|---|
| 0 | — | Configuration and data loading — given |
| 1 | — | Activations and cost functions — **TODO** |
| 2 | — | `ThreeLayerNN` — **TODO** |
| 3 | 2 | Learning rate schedules — **TODO** |
| 4 | — | `train_fold` — **TODO** |
| 5 | — | Cross-validation driver, plotting, summary — given |
| 6 | 1, 2 | Five strategies with the sigmoid output |
| 7 | 3 | Model evaluation and test set — **TODO** |
| 8 | 4 | Softmax with cross-entropy — **TODO** |
| 9 | 1–4 | Written discussion — **TODO** |

Expect the full notebook to take several minutes to run.

In [ ]:
# ============================================================
# Imports (DO NOT MODIFY)
# ============================================================
import warnings

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_digits
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.metrics import confusion_matrix, accuracy_score

In [ ]:
# ============================================================
# Configuration (DO NOT MODIFY)
# ============================================================
SEED        = 42
N_CLASSES   = 10
LAYER_SIZES = (64, 32, 16, 10)
EPOCHS      = 100
N_FOLDS     = 5

# Part 1: fixed learning rates
FIXED_LRS   = [0.01, 0.05, 0.25]

# Part 2: schedule hyperparameters (given)
POWER_ETA0, POWER_C, POWER_S = 0.10, 1.0, 11500
EXP_ETA0,   EXP_S            = 0.10, 57500

np.random.seed(SEED)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
# ============================================================
# Data Loading and Preprocessing (DO NOT MODIFY)
# ============================================================
# The test set is split off once here and must not be touched again until the
# final evaluation at the end of the notebook. Everything in between - training,
# validation, model selection - uses only X_pool / y_pool.
digits = load_digits()
X = digits.data.astype(np.float64) / 16.0     # pixel intensities are 0..16
y = digits.target.astype(int)

splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
pool_idx, test_idx = next(splitter.split(X, y))
X_pool, y_pool = X[pool_idx], y[pool_idx]
X_test, y_test = X[test_idx], y[test_idx]


def one_hot(labels, num_classes=N_CLASSES):
    out = np.zeros((labels.shape[0], num_classes), dtype=np.float64)
    out[np.arange(labels.shape[0]), labels] = 1.0
    return out


def nanmean(a, axis=None):
    # A diverged fold records NaN for its remaining epochs; NaN is the right
    # answer for those, so the all-NaN warning is silenced rather than avoided.
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmean(a, axis=axis)


def nanstd(a, axis=None):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanstd(a, axis=axis)


print(f"CV pool : {X_pool.shape[0]} samples, {X_pool.shape[1]} features")
print(f"Test    : {X_test.shape[0]} samples (held out)")
print(f"Classes : {N_CLASSES}, class counts in pool: {np.bincount(y_pool)}")

## 1. Activation and cost functions

$$J_{\text{SSE}} = \frac{1}{N}\sum_{i=1}^{N}\frac{1}{2}\sum_{k=1}^{10}\left(d_{ik}-y_{ik}\right)^2
\qquad
J_{\text{CE}} = -\frac{1}{N}\sum_{i=1}^{N}\sum_{k=1}^{10} d_{ik}\log y_{ik}$$

where $d_{ik}$ is the desired output and $y_{ik}$ the network output for sample $i$, neuron $k$.
Both costs are averaged over samples, as specified in the assignment. Each returns a single float.

Three numerical points worth handling now rather than debugging later:

- `sigmoid` overflows for large negative induced local fields $v$. Clip the argument.
- `softmax` overflows for large positive $v$. Subtract the row maximum before exponentiating —
  this does not change the result.
- `cost_ce` takes the log of a probability that can underflow to exactly 0. Clip it away from zero.

In [ ]:
# ============================================================
# Activation Functions and Costs (TODO)
# ============================================================
# TODO: implement ReLU and its derivative
def relu(v):
    pass


def relu_deriv(v):
    pass


# TODO: implement the sigmoid function (numerically stable)
def sigmoid(v):
    pass


# TODO: implement the softmax function (numerically stable, row-wise)
def softmax(v):
    pass


# TODO: implement the scaled sum-of-squared-error cost
#       mean over samples of 0.5 * sum_k (d - y)^2
#       Returns: a single float
def cost_sse(y_out, D):
    pass


# TODO: implement the cross-entropy cost
#       mean over samples of -sum_k d * log(y)
#       Returns: a single float
def cost_ce(y_out, D):
    pass


# (DO NOT MODIFY) Dispatch table used by the training loop
COST_FN = {"sigmoid_sse": cost_sse, "softmax_ce": cost_ce}

## 2. The network

One class supports both output layers, selected by `mode`:

| `mode` | output activation | cost |
|---|---|---|
| `"sigmoid_sse"` | sigmoid | scaled sum of squared error |
| `"softmax_ce"` | softmax | cross-entropy |

The output-layer error term $\partial J/\partial v_k$ differs between the two. Part 3(a) of the
assignment asks you to derive it for the softmax case — do that derivation before you write the
code, and use your own result here.

**Initialization.** Draw the weights of each layer from $U(-a, a)$ with
$a = \sqrt{6/(n_{\text{in}} + n_{\text{out}})}$, and set all biases to zero.

**Shapes.** `forward` is called with a single sample of shape `(1, 64)` during training and with a
full matrix of shape `(N, 64)` during evaluation. Write it so both work — no Python loops over
samples inside `forward`.

`get_weights` and `set_weights` are provided. The training loop uses them to snapshot weights, so
do not change how they work.

In [ ]:
# ============================================================
# Three-Layer Neural Network (TODO)
# ============================================================
class ThreeLayerNN:
    def __init__(self, sizes=LAYER_SIZES, mode="sigmoid_sse", seed=0):
        assert mode in ("sigmoid_sse", "softmax_ce")
        self.mode = mode
        rng = np.random.default_rng(seed)
        # TODO: initialize self.W (list of 3 weight matrices) and self.b (list of
        #       3 bias row-vectors). Weights ~ U(-a, a) with a = sqrt(6/(n_in+n_out));
        #       biases all zero. Shapes: W[i] is (sizes[i], sizes[i+1]),
        #       b[i] is (1, sizes[i+1]).
        pass

    def forward(self, x):
        # TODO: implement forward propagation.
        #       Hidden layers use ReLU. The output layer uses sigmoid when
        #       self.mode == "sigmoid_sse" and softmax when "softmax_ce".
        # Returns: (y3, cache) where y3 is the network output of shape (N, 10)
        #          and cache holds whatever backward() needs.
        pass

    def backward(self, cache, D):
        # TODO: implement backpropagation.
        #       The output delta depends on self.mode. Derive it (Part 3(a)) rather
        #       than guessing; the two modes do not share the same expression.
        # Returns: (gW, gb), each a list of 3 arrays matching the shapes of
        #          self.W and self.b.
        pass

    def sgd_step(self, gW, gb, eta):
        # TODO: update self.W and self.b in place using the gradients and
        #       learning rate eta.
        pass

    def predict(self, X):
        # TODO: return predicted class labels, shape (N,)
        pass

    # ---- provided (DO NOT MODIFY) ----
    def get_weights(self):
        return ([w.copy() for w in self.W], [b.copy() for b in self.b])

    def set_weights(self, snap):
        self.W = [w.copy() for w in snap[0]]
        self.b = [b.copy() for b in snap[1]]

## 3. Learning-rate schedules

$n$ is the number of **weight updates**, not the number of epochs. With 1150 training samples per
fold and per-sample SGD, one epoch is 1150 updates.

$$\eta_{\text{power}}(n) = \frac{\eta_0}{\left(1 + n/s\right)^{c}}
\qquad
\eta_{\text{exp}}(n) = \eta_0 \cdot 0.1^{\,n/s}$$

`lr_fixed` is a factory: it takes a learning rate and returns a function of `n`, so that all three
schedules have the same interface and the training loop can call `lr_fn(n)` without knowing which
one it has.

The last cell in this section prints $\eta$ at several values of $n$. Part 2 of the assignment
asks you to report these values.

In [ ]:
# ============================================================
# Learning Rate Schedules (TODO)
# ============================================================
# TODO: return a function of n that always gives the same learning rate
def lr_fixed(eta):
    pass


# TODO: implement power scheduling using POWER_ETA0, POWER_C, POWER_S
def lr_power(n):
    pass


# TODO: implement exponential scheduling using EXP_ETA0, EXP_S
def lr_exponential(n):
    pass

In [ ]:
# ============================================================
# Schedule sanity check (DO NOT MODIFY)
# ============================================================
STRATEGIES = [(f"Fixed eta={eta}", lr_fixed(eta)) for eta in FIXED_LRS] + [
    ("Power schedule",       lr_power),
    ("Exponential schedule", lr_exponential),
]

updates_per_epoch = int(np.ceil(len(X_pool) * (N_FOLDS - 1) / N_FOLDS))
total_updates = updates_per_epoch * EPOCHS
print(f"updates/epoch = {updates_per_epoch}, total updates = {total_updates}\n")
print(f"{'n':>8}  {'power':>10}  {'exponential':>12}")
for n in [0, POWER_S, EXP_S, total_updates]:
    print(f"{n:>8}  {lr_power(n):>10.5f}  {lr_exponential(n):>12.3e}")

## 4. Training loop

`train_fold` trains one model on one fold. Read the contract below carefully — `run_cv`, the
plotting functions and the summary table all depend on it returning exactly these keys.

**Required behaviour**

1. Train for exactly `epochs` epochs. No early stopping.
2. Reshuffle the training set at the start of every epoch.
3. Update the weights after every **individual sample** (per-sample SGD), passing the current
   update count `n` to `lr_fn`.
4. After each epoch, record training and validation cost **and** accuracy.
5. Track the lowest validation cost seen so far and snapshot the weights at that epoch. At the end,
   restore those weights into the model before returning it.
6. Check `weights_are_finite(model)` at the end of each epoch. Weights that have become non-finite
   cannot recover, so stop that run and leave the remaining epochs as `NaN`.

**Why the `np.errstate` wrapper is provided.** Some BLAS builds (Apple Accelerate on macOS in
particular) raise floating-point status flags inside `matmul` even on finite data, so warnings are
not a reliable signal of anything. The wrapper suppresses them; step 6 is how you detect an actual
problem.

**Return value**

| key | type | meaning |
|---|---|---|
| `train_cost`, `val_cost` | `np.ndarray` length `epochs` | per-epoch cost, `NaN` for epochs not reached |
| `train_acc`, `val_acc` | `np.ndarray` length `epochs` | per-epoch accuracy, `NaN` for epochs not reached |
| `best_val_cost` | float | lowest validation cost reached |
| `best_epoch` | int or `NaN` | 1-indexed epoch where that occurred |
| `best_val_acc` | float | validation accuracy at that epoch |
| `diverged_at` | int or `None` | 1-indexed epoch where weights went non-finite |
| `model` | `ThreeLayerNN` | with the best weights restored |

In [ ]:
# ============================================================
# Training Loop (TODO)
# ============================================================
def weights_are_finite(model):
    """(DO NOT MODIFY) True if every weight and bias is finite."""
    return all(np.all(np.isfinite(w)) for w in model.W) and \
           all(np.all(np.isfinite(b)) for b in model.b)


def train_fold(X_tr, y_tr, X_va, y_va, lr_fn, mode, seed=0, epochs=EPOCHS):
    model = ThreeLayerNN(mode=mode, seed=seed)
    rng = np.random.default_rng(seed)
    cost_fn = COST_FN[mode]
    D_tr, D_va = one_hot(y_tr), one_hot(y_va)

    keys = ("train_cost", "val_cost", "train_acc", "val_acc")
    hist = {k: np.full(epochs, np.nan) for k in keys}
    best = {"cost": np.inf, "epoch": None, "weights": model.get_weights()}
    n = 0
    diverged_at = None

    with np.errstate(over="ignore", invalid="ignore", divide="ignore"):
        for epoch in range(epochs):
            # TODO: one epoch of per-sample SGD.
            #       - reshuffle the training indices
            #       - for each sample: eta = lr_fn(n), forward, backward,
            #         sgd_step, then increment n
            pass

            # TODO: stop this run if the weights are no longer finite.
            #       Call weights_are_finite(model), defined just above. If it
            #       returns False, set diverged_at = epoch + 1 and `break`.
            #       Nothing else is needed: hist was created with np.full(epochs,
            #       np.nan), so every epoch you never reach is already NaN. The
            #       nanmean / nanstd helpers then skip those entries when the
            #       curves are averaged across folds.

            # TODO: run the whole training and validation sets through
            #       model.forward() and record this epoch's results into
            #       hist["train_cost"][epoch], hist["val_cost"][epoch],
            #       hist["train_acc"][epoch], hist["val_acc"][epoch].
            #       Use cost_fn (set from COST_FN above) for the costs, and
            #       compare np.argmax(output, axis=1) against the labels for
            #       the accuracies.

            # TODO: if hist["val_cost"][epoch] is the lowest validation cost so
            #       far, update `best` with that cost, the 1-indexed epoch
            #       (epoch + 1), and model.get_weights()

    # ---- provided (DO NOT MODIFY) ----
    hist["diverged_at"]   = diverged_at
    hist["best_val_cost"] = best["cost"]
    hist["best_epoch"]    = best["epoch"] if best["epoch"] is not None else np.nan
    hist["best_val_acc"]  = (float(hist["val_acc"][best["epoch"] - 1])
                             if best["epoch"] is not None else np.nan)
    model.set_weights(best["weights"])
    hist["model"] = model
    return hist

In [ ]:
# ============================================================
# Cross-Validation Driver (DO NOT MODIFY)
# ============================================================
def run_cv(name, lr_fn, mode, n_folds=N_FOLDS):
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=SEED)
    folds = []
    for f, (tr, va) in enumerate(skf.split(X_pool, y_pool)):
        folds.append(train_fold(X_pool[tr], y_pool[tr],
                                X_pool[va], y_pool[va],
                                lr_fn, mode, seed=f))
    return {
        "name": name, "mode": mode, "folds": folds,
        "train_cost": np.vstack([f["train_cost"] for f in folds]),
        "val_cost":   np.vstack([f["val_cost"]   for f in folds]),
        "train_acc":  np.vstack([f["train_acc"]  for f in folds]),
        "val_acc":    np.vstack([f["val_acc"]    for f in folds]),
        "best_val_costs": np.array([f["best_val_cost"] for f in folds], dtype=float),
        "best_val_accs":  np.array([f["best_val_acc"]  for f in folds], dtype=float),
        "best_epochs":    np.array([f["best_epoch"]    for f in folds], dtype=float),
        "diverged":       [f["diverged_at"] for f in folds],
    }

In [ ]:
# ============================================================
# Plotting and Summary Helpers (DO NOT MODIFY)
# ============================================================
def _band(ax, curves, label, color):
    e = np.arange(1, curves.shape[1] + 1)
    m, s = nanmean(curves, axis=0), nanstd(curves, axis=0)
    ax.plot(e, m, label=label, color=color)
    ax.fill_between(e, m - s, m + s, alpha=0.20, color=color)


def plot_cv_curves(res):
    cost_label = ("Cost: mean 0.5*SSE" if res["mode"] == "sigmoid_sse"
                  else "Cost: mean cross-entropy")
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    _band(ax[0], res["train_cost"], "Train", "tab:blue")
    _band(ax[0], res["val_cost"],   "Validation", "tab:orange")
    ax[0].set_yscale("log"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel(cost_label)
    ax[0].set_title(f"Cost - {res['name']}"); ax[0].legend(); ax[0].grid(alpha=0.3)

    _band(ax[1], res["train_acc"], "Train", "tab:blue")
    _band(ax[1], res["val_acc"],   "Validation", "tab:orange")
    ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Accuracy")
    ax[1].set_title(f"Accuracy - {res['name']}"); ax[1].legend(); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()


def plot_comparison(results, title):
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    for r in results:
        e = np.arange(1, r["val_cost"].shape[1] + 1)
        ax[0].plot(e, nanmean(r["val_cost"], axis=0), label=r["name"])
        ax[1].plot(e, nanmean(r["val_acc"],  axis=0), label=r["name"])
    ax[0].set_yscale("log"); ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Mean validation cost")
    ax[0].set_title(f"Validation cost - {title}"); ax[0].legend(fontsize=8); ax[0].grid(alpha=0.3)
    ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Mean validation accuracy")
    ax[1].set_title(f"Validation accuracy - {title}"); ax[1].legend(fontsize=8); ax[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()


def summarize(results):
    print(f"{'strategy':<24}{'val cost (mean+/-std)':<26}{'val acc (mean+/-std)':<26}{'best epoch':>12}")
    print("-" * 88)
    for r in results:
        c, a, e = r["best_val_costs"], r["best_val_accs"], r["best_epochs"]
        n_div = sum(d is not None for d in r["diverged"])
        note = f"  [{n_div}/{len(r['diverged'])} folds diverged]" if n_div else ""
        print(f"{r['name']:<24}{nanmean(c):.4f} +/- {nanstd(c):.4f}{'':<9}"
              f"{nanmean(a):.4f} +/- {nanstd(a):.4f}{'':<9}"
              f"{int(np.nanmedian(e)):>12}{note}")


def peak_val_acc(res):
    """Peak of the fold-averaged validation accuracy curve, and its 1-indexed epoch."""
    curve = nanmean(res["val_acc"], axis=0)
    return float(np.nanmax(curve)), int(np.nanargmax(curve)) + 1

## 5. Parts 1 and 2 — the five learning rate strategies

Each of the three fixed rates and the two schedules is run through the full 5-fold
cross-validation with the sigmoid output and the scaled sum of squared error.

In [ ]:
# ============================================================
# Sigmoid + SSE experiments (DO NOT MODIFY)
# ============================================================
results_sse = []
for name, fn in STRATEGIES:
    r = run_cv(name, fn, mode="sigmoid_sse")
    results_sse.append(r)
    n_div = sum(d is not None for d in r["diverged"])
    note = f"   <- diverged in {n_div}/{N_FOLDS} folds" if n_div else ""
    print(f"{name:<24} val cost {nanmean(r['best_val_costs']):.4f}  "
          f"val acc {nanmean(r['best_val_accs']):.4f}  "
          f"median best epoch {int(np.nanmedian(r['best_epochs']))}{note}")

In [ ]:
# (DO NOT MODIFY)
for r in results_sse:
    plot_cv_curves(r)

In [ ]:
# (DO NOT MODIFY)
plot_comparison(results_sse, "sigmoid + SSE")
print()
summarize(results_sse)

## 6. Part 3 — model evaluation

Three things happen here, in order:

1. **3(b)** — for *each* of the five strategies, the epoch of lowest mean validation cost and the
   epoch of highest mean validation accuracy.
2. **3(a)** — select one strategy using the mean cross-validated validation accuracy.
3. **3(c)–3(e)** — evaluate the five fold checkpoints of the selected strategy on the test set, and
   plot the confusion matrix summed over them.

In [ ]:
# ============================================================
# Part 3(b) - per-strategy epochs (TODO)
# ============================================================
# TODO: for EACH of the five strategies in results_sse, print
#         - the epoch at which the fold-mean validation COST is lowest
#         - the epoch at which the fold-mean validation ACCURACY is highest
#       Use nanmean(r["val_cost"], axis=0) and nanmean(r["val_acc"], axis=0),
#       then np.nanargmin / np.nanargmax. Remember the curves are 0-indexed
#       but epochs are reported 1-indexed.
#
#       Print one row per strategy with both epochs side by side, so the
#       relationship asked about in the discussion is visible at a glance.
print(f"{'strategy':<24}{'epoch of min val cost':>24}{'epoch of max val acc':>24}")
print("-" * 72)

pass

In [ ]:
# ============================================================
# Part 3(a) - model selection (TODO)
# ============================================================
# Selection uses the mean cross-validated validation ACCURACY. peak_val_acc(r)
# is provided and returns (peak_accuracy, epoch_of_peak).
#
# TODO: rank the five strategies in results_sse by peak mean CV validation
#       accuracy, highest first, and print the ranking.


# TODO: set the four variables below.
#         overall      - the selected result dict from results_sse
#         final_epochs - epoch of peak mean validation accuracy
#         cv_acc       - the peak mean CV validation accuracy
#         cv_std       - std across the 5 folds of val_acc at that epoch
overall      = None
final_epochs = None
cv_acc       = None
cv_std       = None

print(f"\nSelected: {overall['name']}")
print(f"  mean CV validation accuracy : {cv_acc:.4f} +/- {cv_std:.4f}")
print(f"  epoch of peak val accuracy  : {final_epochs}")

### Part 3(c)–3(d) — test evaluation

Cross-validation has already trained five models for the selected strategy, one per fold, and
`train_fold` restored each one's **checkpointed weights** (those at that fold's lowest validation
cost) before returning it. They are available as `overall["folds"][i]["model"]`.

Evaluate all five on the test set and report the mean and standard deviation. Do not retrain, and
do not pick a single fold — every fold model is used.

Note what the standard deviation does and does not capture; part 3 asks you to judge whether the
differences you see are meaningful.

In [ ]:
# ============================================================
# Part 3(c)-3(d) - test evaluation of the five fold models (TODO)
# ============================================================
# TODO: for each of the five folds of `overall`, take its checkpointed model
#       (overall["folds"][i]["model"]) and compute its accuracy on the test set
#       with accuracy_score. Collect them into test_accs.
test_accs = None

# TODO: print one line per fold showing that fold's validation accuracy
#       (overall["folds"][i]["best_val_acc"]) next to its test accuracy.


print(f"\nTest accuracy over the 5 fold models: "
      f"{np.mean(test_accs):.4f} +/- {np.std(test_accs):.4f}")
print(f"CV validation accuracy               : {cv_acc:.4f} +/- {cv_std:.4f}")

In [ ]:
# ============================================================
# Part 3(e) - Confusion Matrix (TODO)
# ============================================================
# TODO: build ONE confusion matrix summed over the five fold models: compute
#       each fold model's confusion matrix on the test set with
#       confusion_matrix(..., labels=range(N_CLASSES)) and add them elementwise.
#       The result covers 5 x 360 = 1800 predictions.
cm = None

# TODO: plot cm as a seaborn heatmap with integer annotations, labelled axes
#       and a title.


# TODO: print the most frequent off-diagonal confusions as
#       "true -> predicted : count", sorted by count descending.

## 7. Part 4 — softmax output with cross-entropy

Same hidden layers, same training loop, different output layer. Part 4(a) asks you to derive the
output-layer error term for this combination before you implement it — do the derivation first and
use your own result in `backward`.

Only the **best strategy identified in part 3** is used here, over the same five folds, and the
five checkpoints are evaluated on the test set exactly as in part 3.

### Part 4(a) — derivation (TODO)

**Replace this cell with your derivation.** Required content:

1. $\partial J / \partial v_k$ for the softmax output with the cross-entropy cost, showing your work.
2. The gradient with respect to the output-layer weights, written out.
3. The corresponding error term for the sigmoid output with the scaled sum of squared error.
4. An account of the difference between the two expressions.

You may write this in LaTeX inside this markdown cell.

In [ ]:
# ============================================================
# Part 4(b)-4(c) - softmax network at the selected strategy (TODO)
# ============================================================
# TODO: run run_cv() once, with mode="softmax_ce" and the learning rate
#       strategy selected in part 3 (overall["name"]). Store it in result_ce.
#       Print the same per-configuration summary line as in section 5:
#       mean best validation cost, mean best validation accuracy, and the
#       median best epoch.
result_ce = None

pass

In [ ]:
# (DO NOT MODIFY) Learning curves for the softmax configuration
plot_cv_curves(result_ce)
print()
summarize([result_ce])

In [ ]:
# ============================================================
# Part 4 - softmax: epochs and test evaluation (TODO)
# ============================================================
# The same protocol as part 3, on the same five folds, so the two output
# layers are compared on identical splits.
#
# TODO: print the epoch of lowest mean validation cost and the epoch of highest
#       mean validation accuracy for result_ce, as in Part 3(b).


# TODO: set ce_cv_acc and ce_cv_std from result_ce, the same way cv_acc and
#       cv_std were set in Part 3(a).
ce_cv_acc = None
ce_cv_std = None

# TODO: evaluate the five softmax fold checkpoints on the test set and collect
#       the accuracies into ce_test_accs. Print one line per fold, as in part 3.
ce_test_accs = None


print(f"\nSoftmax test accuracy over the 5 fold models: "
      f"{np.mean(ce_test_accs):.4f} +/- {np.std(ce_test_accs):.4f}")
print(f"Sigmoid test accuracy over the 5 fold models: "
      f"{np.mean(test_accs):.4f} +/- {np.std(test_accs):.4f}")

## 8. Written Discussion (REQUIRED)

Failure to include this discussion will result in point deductions. Answer each item in a markdown
cell below, referring to your own plots and printed numbers.

**Part 1 — fixed learning rates.** Discuss how η = 0.01, 0.05 and 0.25 each influence convergence.
Analyze the learning curves for the training and validation sets, note any patterns or differences,
and state what evidence in the curves supports your conclusions.

**Part 2 — schedules.** Report the η values printed by the schedule cell at n = 0, n = s and
n = n_total. Compare both schedules against each other and against the three fixed rates, and
discuss their effect on convergence and on the learning curves.

**Part 3(a) — selection criterion.** Which is the better selection criterion, validation accuracy or
validation cost? Justify your answer.

**Part 3(b) — epochs.** Discuss the relationship between the epoch of lowest validation cost and the
epoch of highest validation accuracy, using the per-strategy table you printed.

**Part 3(e) — confusion matrix.** Which digit pairs account for most of the remaining errors, and
why are those particular pairs confused?

**Part 3 — generalization.** Compare training, cross-validated validation, and test results. Are the
differences large enough to be meaningful, given the size of the test set?

**Part 4(a) — derivation.** (Answered in the derivation cell above.)

**Part 4(d) — output layers.** Compare the two output layers on speed of convergence and on
sensitivity to the choice of learning rate. Relate what you observe to the expressions you derived
in part 4(a).